# case145 / case300 powerflow via the native MATPOWER reader

The CIM exports of these two cases (examples/Notebooks/Grids/case145.ipynb, case300.ipynb) are skipped: their third-party conversion misclassifies some transformers as plain lines, wiring together buses at different voltage levels. `dpsim.matpower.Reader` reads the MATPOWER `mpc` struct directly (bus/branch/gen matrices, including the branch `ratio` column) and classifies lines vs transformers correctly, so it isn't affected. This notebook runs both cases through that reader and checks the solved voltages against MATPOWER's own solution.

In [ ]:
import os
import sys
import subprocess
import urllib.request

dpsim_root_dir = (
    subprocess.Popen(["git", "rev-parse", "--show-toplevel"], stdout=subprocess.PIPE)
    .communicate()[0]
    .rstrip()
    .decode("utf-8")
)
sys.path.insert(0, os.path.join(dpsim_root_dir, "python/src/dpsim/"))

import matpower
import dpsimpy
import numpy as np
from villas.dataprocessing.readtools import read_timeseries_csv

In [ ]:
base_url = "https://raw.githubusercontent.com/dpsim-simulator/reference-results/master/Matpower/"
cases = ["case145results", "case300results"]
for case in cases:
    filename = case + ".mat"
    if not os.path.exists(filename):
        urllib.request.urlretrieve(base_url + filename, filename)

In [ ]:
def run_case(case):
    mpc_reader = matpower.Reader(mpc_file_path=case + ".mat", mpc_name=case)
    mpc_reader.load_mpc(domain=matpower.Domain.PF)
    system = mpc_reader.system

    dpsimpy.Logger.set_log_dir("logs/" + case)
    logger = dpsimpy.Logger(case)
    for node in system.nodes:
        logger.log_attribute(node.name() + ".V", "v", node)

    sim = dpsimpy.Simulation(case, dpsimpy.LogLevel.off)
    sim.set_system(system)
    sim.set_time_step(0.1)
    sim.set_final_time(0.1)
    sim.set_domain(dpsimpy.Domain.SP)
    sim.set_solver(dpsimpy.Solver.NRP)
    sim.do_init_from_nodes_and_terminals(False)
    sim.set_solver_component_behaviour(dpsimpy.SolverBehaviour.Initialization)
    sim.add_logger(logger)
    sim.run()

    ts = read_timeseries_csv("logs/" + case + "/" + case + ".csv")
    mp_results = mpc_reader.get_pf_results()

    bus_data = mpc_reader.mpc_bus_data.set_index("bus_i")
    vm_dpsim = np.array(
        [
            abs(ts["N" + str(bus_i) + ".V"].values[-1])
            / (bus_data.at[bus_i, "baseKV"] * 1e3)
            for bus_i in bus_data.index
        ]
    )
    vm_matpower = mp_results["Vm [pu]"].values.astype(float)

    return len(system.nodes), np.abs(vm_dpsim - vm_matpower)

In [ ]:
import pandas as pd

rows = []
for case in cases:
    buses, vm_diff = run_case(case)
    rows.append(
        {
            "case": case,
            "buses": buses,
            "max |Vm diff| [pu]": vm_diff.max(),
            "mean |Vm diff| [pu]": vm_diff.mean(),
        }
    )

results = pd.DataFrame(rows).set_index("case")
results

In [ ]:
# DPsim's Newton-Raphson and MATPOWER's own solver converge to slightly
# different points within their respective tolerances; this only needs to
# rule out a structurally wrong solve (the kind the broken CIM data produced).
assert (results["max |Vm diff| [pu]"] < 0.02).all()